# Arducam Sensor Explore

There is very little information on the sensors from Arducam. This notebook explores some of the relevant registers and settings of the board.

Note that these observations were made with MT9J001/003 on board UC593 Rev.C and may be not valid on other setups.

In [ ]:
import os, sys, time
from pathlib import Path
print(sys.version_info)

import arducam_config_parser
import ArducamSDK

import numpy as np
import cv2
from IPython.display import Image
import matplotlib.pyplot as plt

handle = None

In [ ]:
# Pick a configuration file with GUI
# %gui tk
# import tkinter
from tkinter.filedialog import askopenfilename
configName = askopenfilename() # show an "Open" dialog box and return the path to the selected file
# %gui
print("Config file:", configName, type(configName), bool(configName))

In [ ]:
## All configuration files for MT9J001/2 from the main repo.
# configName = Path('../Config') / 'MT9J001_MONO_8b_640x480.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_640x480.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_640x480.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_640x480.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_1280x720.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_1280x720.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_1280x720.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_1280x720.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_1920x1080.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_1920x1080.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_1920x1080.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_1920x1080.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x2748.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_3664x2748.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_3664x2748.cfg'    # Error, pll_multiplier set to 0x90, over maximum 0x80
# configName = Path('../Config') / 'MT9J003_RAW_12b_3664x2748.cfg'   # Error, needs active VRCMD = 0xD7, 0x4600, 0x0100, 1, 0x85

# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x512.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x64.cfg'

configName = Path('../Config') / 'test.cfg'


In [ ]:
if handle is not None:
    ArducamSDK.Py_ArduCam_close(handle)

handle = ArducamSDK.initCameraFromFile(configName)

## Capture Image

In [ ]:
ArducamSDK.Py_ArduCam_setMode(handle,ArducamSDK.CONTINUOUS_MODE)
rtn_val = ArducamSDK.Py_ArduCam_beginCaptureImage(handle)
print("Begin Capture: ",hex(rtn_val))
for i in range(5):
    time0 = time.time()
    rtn_val = ArducamSDK.Py_ArduCam_captureImage(handle)
    if rtn_val<255:
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            err_code, data, img_cfg = ArducamSDK.Py_ArduCam_readImage(handle)
            time1 = time.time()
            dt = time1-time0
            print(f"       Image Captured: {err_code:#06X},    time: {dt:9.6f}, fps: {1/dt:.3f}")
            ArducamSDK.Py_ArduCam_del(handle)
            break
    else:
        # an image may be retrieved even if an error code is returnedm but may be incomplete
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            ArducamSDK.Py_ArduCam_del(handle)
        
    print(f"{i:3}    Capture Image: {rtn_val:#06X}    Available images: {ArducamSDK.Py_ArduCam_availableImage(handle)}")

ArducamSDK.Py_ArduCam_flush(handle)
rtn_val = ArducamSDK.Py_ArduCam_endCaptureImage(handle)
print("End Capture: ",hex(rtn_val))


In [ ]:
if len(data) == img_cfg['u32Height']*img_cfg['u32Width']:
    image = np.frombuffer(data, np.uint8).reshape(img_cfg['u32Height'], img_cfg['u32Width'])
else:
    image = (np.frombuffer(data, np.uint16).reshape(img_cfg['u32Height'], img_cfg['u32Width']) >> (img_cfg['u8PixelBits']-8))
    image = image.astype(np.uint8)
    
err_code, rgb_mode = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3024)
print(f'{hex(rgb_mode)=}')
# image = cv2.cvtColor(image, cv2.COLOR_BayerRG2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGR2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGB2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerBG2BGR)

# Encode the array to PNG and show it directly with IPython
retval, buffer = cv2.imencode('.png', image)
# Image(data=buffer)
Image(data=buffer, width=1000)

In [ ]:
plt.imshow(image[1300:1500,1500:2000], aspect=1)
# plt.axis('off')

In [ ]:
plt.imshow(image, aspect=1)
# plt.axis('off')

In [ ]:
if handle is not None:
    ArducamSDK.Py_ArduCam_close(handle)
handle = None